# Case Study: Model Parametrization Risk in a Yield-Curve Engine
### 4-Parameter Nelson-Siegel-Svensson vs. 3-Parameter Nelson-Siegel — What a Diagnostic Failure Actually Costs a Bank

This notebook documents a real model-risk investigation that started from a single suspicious number in a
diagnostic cross-check, traced it to its root cause, fixed it, and then measured — honestly, with real
numbers — what the fix actually changes on a bank's balance sheet and income statement. It is written as a
narrative case study: no code is executed here, but every number quoted comes from code that *was* executed
(the companion notebook `term_structure_irrbb_engine_3param_NS.ipynb`), and the relevant snippets are
reproduced inline so the logic is auditable without re-running anything.


## 1. Background — what the model does and why it matters

A bank's ALM (Asset-Liability Management) desk needs a smooth, arbitrage-free zero-coupon yield curve to:

- **Discount every cash flow** on the banking book to compute Economic Value of Equity (EVE)
- **Decompose** curve risk into a small number of interpretable factors (level, slope, curvature) so that risk
  can be hedged and reported factor-by-factor, not instrument-by-instrument
- **Price internal Funds Transfer (FTP)** rates for behavioural products like non-maturing deposits (NMDs)
- **Feed relative-value trading signals** (e.g. a 2s10s steepener) that depend on how the curve's shape moves
  day to day

The standard academic and industry tool for this is the **Nelson-Siegel family** of parametric curve models.
The variant used in the original notebook was the **4-parameter Nelson-Siegel-Svensson (NSS)** extension.

### The variables, defined

| Symbol | Name | Role |
|---|---|---|
| β0 | Level | The long-run/asymptotic level of the curve. Should move almost 1:1 with a broad parallel shift. |
| β1 | Slope | Short-vs-long spread. Governs how steep or flat the curve is. |
| β2 | Curvature 1 | A hump/trough term, typically peaking around the belly of the curve (3-7Y). |
| β3 | Curvature 2 (Svensson extension only) | A second hump/trough term, meant to capture a *separate* medium/long-end shape feature the first curvature term misses. |
| λ1 | Decay 1 | Controls where curvature1's hump is centered (in years). |
| λ2 | Decay 2 (Svensson extension only) | Controls where curvature2's hump is centered. |

Each parameter is the coefficient on a **loading column** in a linear design matrix; for a given tenor grid
(here 1Y, 2Y, 3Y, 5Y, 7Y, 10Y, 15Y, 20Y, 30Y), fitting the curve on any single day is an ordinary least-squares
(OLS) regression of observed yields on these four (or three) loading columns, with λ1/λ2 fixed globally and
β0-β3 re-estimated fresh every day.


## 2. The diagnostic that raised the flag

The notebook includes a standard model-validation cross-check: **daily changes in the fitted level factor
(Δβ0) should correlate strongly with the first principal component (PC1) of daily changes in the observed
curve itself** — because both are, in theory, measuring the same thing: "how much did the whole curve move
today." In the finance literature (Diebold-Li and related work) this correlation is typically **above 0.9**.

Running the cross-check on the notebook's actual global calibration (fitted on 3 years of real Federal Reserve
GSW/SVENY continuously-compounded zero-coupon data) gave:

```python
beta0_change = nss_params["beta0"].diff().reindex(daily_changes.index)
level_corr = np.corrcoef(pca_scores[:, 0], beta0_change.values)[0, 1]
print(f"corr(PC1 score, Δbeta0): {level_corr:+.3f} (sign is arbitrary)")
```

**Result: `corr(PC1 score, Δbeta0): +0.172`** — essentially broken. The level factor the model calls "β0" was
not, in practice, measuring the same thing as the data's actual dominant mode of variation.


## 3. Root-cause diagnosis

The global lambda calibration (multi-start L-BFGS-B on 3 years of daily data) had converged to:

```
Global lambda1 / lambda2 : 2.4453 / 15.4749 years
Design condition number  : 341.0
```

λ2 = 15.47 years is the tell. The notebook's tenor grid tops out at **30Y**, so for the longest observed
point, `τ/λ2 = 30/15.47 ≈ 1.94` — the second curvature term never completes even two "half-lives" within the
observed data. A loading column that barely varies across the observed tenor range is, numerically, close to
redundant with whichever other column shares its rough shape.

Computing the design matrix's column correlations at the fitted λ's confirmed it directly:

| | slope | curvature1 | curvature2 |
|---|---|---|---|
| **slope** | 1.000 | 0.441 | **-0.984** |
| **curvature1** | 0.441 | 1.000 | -0.591 |
| **curvature2** | -0.984 | -0.591 | 1.000 |

`corr(slope, curvature2) = -0.984` — the two columns are nearly mirror images of each other. OLS cannot
separate two near-identical regressors: it produces a fitted curve that is accurate (mean RMSE 1.55bp across
the panel), but the *individual* β1/β3 coefficients become unstable and trade off against each other from one
day to the next, and — because the constant (β0) column is estimated jointly with all others in the same
`lstsq` solve — that day-to-day instability leaks into β0 as well, breaking its correspondence with PC1.

**This was confirmed experimentally, not just argued analytically.** Refitting the *same* synthetic panel
(generated from *known* λ1=1.5/λ2=4.5) but forcing the notebook's real-data λ pair (2.445/15.47) reproduced
the exact same condition number (341.0) and collapsed `corr(PC1, Δβ0)` from 0.908 down to 0.172 — proving the
break was caused by the λ2 choice itself, not by anything specific to that day's data.


## 4. The fix — dropping to a classic 3-parameter Nelson-Siegel

The simplest fix that removes the collinearity **structurally**, rather than papering over it with tighter
optimizer bounds, is to drop the second curvature term (and its λ2) entirely and fit the classic
**3-parameter Nelson-Siegel** model instead: β0 (level), β1 (slope), β2 (curvature), one shared λ.

```python
def nss_loadings(tau, lambda1):
    tau = np.asarray(tau, dtype=float)
    tau_safe = np.maximum(tau, 1e-8)
    x1 = tau_safe / lambda1
    level = np.ones_like(tau_safe)
    slope = -np.expm1(-x1) / x1
    curvature = slope - np.exp(-x1)
    return np.column_stack([level, slope, curvature])


def nss_zero_rate_pct(beta, tau, lambda1):
    beta = np.asarray(beta, dtype=float)
    return nss_loadings(np.atleast_1d(tau), lambda1) @ beta
```

Global calibration on the same 3-year panel now optimizes a single λ instead of a λ-pair:

```python
def calibrate_global_lambda(panel, tenors):
    sampled = panel.iloc[::5].values
    def objective(x):
        lambda1 = np.exp(x[0])
        betas, fitted, condition_number = panel_fit_for_lambda(sampled, tenors, lambda1)
        mse = np.mean((fitted - sampled) ** 2)
        condition_penalty = 1e-8 * max(condition_number - 1_000.0, 0.0) ** 2
        return mse + condition_penalty
    starts = [0.5, 1.0, 1.5, 2.5, 4.0]
    ...
```


## 5. Did the fix work? — the diagnostic, re-run

| Metric | 4-param Svensson (before) | 3-param Nelson-Siegel (after) |
|---|---|---|
| Global λ | λ1=2.4453, λ2=15.4749 | λ1≈2.41–2.51 (varies slightly by sample) |
| Design condition number | **341.0** | **17–20** |
| corr(slope, curvature2) | -0.984 | *(term removed)* |
| corr(PC1, Δβ0) | **+0.172** (broken) | **+0.67 to +0.97** depending on sample — restored to a defensible range |
| Mean daily RMSE (3-year panel) | 1.55 bp | 3.29 bp |
| 95th pct daily RMSE | 3.38 bp | 6.00 bp |
| Max single-point error (panel) | 7.47 bp | 16.96 bp |

The collinearity is gone — condition number dropped roughly 17-20x, and the level factor now behaves like a
level factor. But this did not come for free: **panel-level fit quality degraded by roughly 2x on RMSE and
2.3x on worst-case point error.** This is a textbook bias-variance trade-off: the discarded curvature2 term
was collinear, but it was not *useless* — it was absorbing real medium/long-end curve shape that the 3-parameter
model can no longer represent.

Note also that the real-data correlation recovery (+0.67 to +0.97) is noisier than a controlled synthetic test
would suggest, because real Fed curve data contains genuine multi-factor variation (FOMC surprises,
belly-specific supply/demand effects) that no 3-factor model — however well conditioned — can fully compress
into a single level factor. A rise from 0.17 to the high 0.6s/0.9s is the honest ceiling here, not a bug.


## 6. What it costs on the EVE side

To find out whether this trade-off actually matters for the bank's regulatory numbers, both fitted curves
were used to value the **same illustrative banking book** (assets, wholesale liabilities, and an NMD
replicating portfolio reconciling to Tier 1 capital) on the same real observed curve date (2026-07-17),
running the full BCBS six-scenario EVE shock suite through each:

| | 4-param Svensson | 3-param Nelson-Siegel | Difference |
|---|---|---|---|
| Single-day curve fit RMSE | 3.47 bp | 4.01 bp | +0.54 bp |
| Base EVE | 209.95 mm | 209.97 mm | **-0.02 mm** |
| Worst scenario | steepener | steepener | same |
| Worst adverse ΔEVE loss | 10.31 mm | 10.35 mm | +0.04 mm |
| Worst adverse ΔEVE / Tier 1 | 6.87% | 6.90% | **+0.03 pp** |
| Outlier breach (≥15%) | No | No | unchanged |

**The single-day valuation and regulatory-ratio impact is negligible.** Both models agree closely because
every instrument in the book has a maturity inside the 1Y-30Y observed tenor range — both curves are
*interpolating*, not extrapolating, at every cash flow date the book actually has. Point-in-time EVE and the
15% Tier 1 outlier test are essentially insensitive to which of the two models is used.


## 7. What it costs on the NII / FTP side

The NMD (non-maturing deposit) replicating portfolio's Funds Transfer Price is read directly off the fitted
curve at the 1Y-5Y core ladder tenors. Comparing the two models at those tenors, on the same day:

| Tenor | 4-param FTP | 3-param FTP | Difference |
|---|---|---|---|
| 1Y | 4.0978% | 4.1044% | -0.67 bp |
| 2Y | 4.0740% | 4.0649% | +0.91 bp |
| 3Y | 4.1070% | 4.0988% | +0.82 bp |
| 4Y | 4.1722% | 4.1711% | +0.11 bp |
| 5Y | 4.2541% | 4.2607% | -0.66 bp |
| **Blended FTP** | **4.1410%** | **4.1400%** | **+0.10 bp** |

On a 585mm core NMD book, a 0.10bp blended FTP difference is roughly **6 thousand currency units per year** —
immaterial for pricing and transfer-pricing decisions.

**Conclusion: neither EVE nor NII/FTP is meaningfully affected by this choice**, for this particular book, on
this particular day, because both are point-in-time valuations inside the well-observed tenor range.


## 8. So where does the cost actually show up?

If the bottom-line valuation is nearly identical either way, is the collinearity finding a non-issue? No —
the cost is real, it is just **not on the balance sheet**. It shows up in three other places:

**1. Factor-based risk attribution and hedging.** Any process that asks "how much of today's P&L came from a
level move vs. a slope move" — which is exactly what a rates desk uses to decide whether to hedge with a
bullet or a barbell — relies on the *individual* β's being stable and interpretable, not just on the fitted
curve being accurate in aggregate. The 4-parameter model's β0/β1/β3 traded off against each other day to day;
a hedge ratio or risk-budget derived from that decomposition on any single day could be materially wrong even
though the priced value of the book that day was fine.

**2. Model governance and validation cost.** A model-risk function running this exact PCA cross-check would
flag `corr(PC1, Δβ0) = 0.172` as a formal validation finding. Remediating a finding — documentation, senior
sign-off, possibly a temporary restriction on model use pending a fix — is a real cost in staff time and
calendar time, independent of whether the point valuation was ever wrong. This is precisely the kind of
"invisible in EVE, very visible in an exam" issue that model-risk teams exist to catch.

**3. Time-series strategies that consume the betas directly.** The notebook's DV01-neutral 2s10s carry-and-roll
signal is built from the *daily path* of the fitted curve, not from a single day's valuation. Day-to-day beta
instability from the 4-parameter model's collinearity can leak into position-sizing logic that depends on
factor decomposition (e.g. hedging slope exposure specifically) even where whole-curve-based DV01 hedges
(as this particular signal actually uses) are less affected.

**4. Tail risk, not observed here.** All of the EVE/NII comparisons above are *inside* the 1Y-30Y observed
range. A well-conditioned vs. poorly-conditioned model can diverge much more sharply when extrapolating beyond
the longest observed tenor (e.g. valuing a 40Y cash flow, or applying a shock scenario that shifts the
effective curve shape into a region neither model was fit against). This case study's book has no such
exposure, so this risk is flagged, not measured.


## 9. Recommendation — which one, and why

| | Choose 4-param Svensson if... | Choose 3-param Nelson-Siegel if... |
|---|---|---|
| Priority | Tightest possible daily fit (lower RMSE) matters most, e.g. for precise bond relative-value trading | Interpretable, stable risk-factor decomposition matters most, e.g. for regulatory reporting, model validation, and factor-based hedging |
| Tenor grid | Long-dated (30Y+) with genuinely rich medium/long-end shape variation | Any grid where the second curvature term would be poorly identified given the observed maximum tenor |
| Audience | Internal desk use only, with no PCA-style cross-checks in the review process | Anything that will face a model validator, regulator, or a portfolio reviewer who will run standard diagnostics |

**For this specific engine — a prototype explicitly built to be shown to an ALM/quant hiring audience and to
pass its own stated validation checks — the 3-parameter choice is the right one.** The EVE/NII cost of doing
so is close to zero for a book like this one (well inside the observed tenor range), while the governance and
interpretability benefit is large and directly falsifiable (the PCA cross-check either passes convincingly or
it doesn't). The 2x degradation in raw curve-fit RMSE is a real, honestly-disclosed cost — but it buys back
something a 4-parameter model with an unconstrained λ2 cannot promise: a level factor that means what it says
it means.

**A middle path exists and is worth flagging for future work:** re-run the 4-parameter calibration with a
tighter upper bound on λ2 (e.g. capping the optimizer search at 8 years instead of 20), which would likely
recover most of the original fit quality while keeping the second curvature term inside its identifiable
region for a 30Y-max tenor grid. This case study tested the two extremes (unconstrained 4-parameter vs. no
second term at all); the constrained-λ2 middle ground was not tested here and is the natural next experiment.
